In [7]:
class AST:
    pass

class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    def __init__(self, value):
        self.value = value

class Var(AST):
    def __init__(self, name):
        self.name = name

class ParserError(Exception):
    pass

In [8]:
import re

class MiniCompiler:
    def __init__(self, source, env):
        self._tokens = iter(re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()\-^]', source) + ['?'])
        self._current = None
        self._env = env 
        self._temp_count = 0
        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (expected == "ID" and self._current.isalnum()):
            raise ParserError(f"Expected {expected}, found {self._current}")
        token = self._current
        self.advance()
        return token

    def factor(self):
        token = self._current
        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))
        elif token and token.isalpha():
            if token not in self._env:
                raise ParserError(f"Semantic Error: Undefined variable '{token}'")
            self.advance()
            return Var(token)
        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node
        raise ParserError(f"Unexpected token: {token}")

    def power(self):
        node = self.factor()
        if self._current == '^':
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.power())
        return node

    def term(self):
        node = self.power() # Ubah ini
        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.power()) # Dan sesuaikan pemanggilan kanannya
        return node

    def expr(self):
        node = self.term()
        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.term())
        return node

    def generate_tac(self, node):
        if isinstance(node, Num): return str(node.value)
        if isinstance(node, Var): return node.name
        
        left_val = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)
        
        self._temp_count += 1
        temp_name = f"t{self._temp_count}"
        print(f"{temp_name} = {left_val} {node.op} {right_val}")
        return temp_name


In [9]:
source_code = "a ^ 2 + b * c"
symbol_table = {'a': 5, 'b': 10, 'c': 2}

try:
    print(f"Input: {source_code}")
    compiler = MiniCompiler(source_code, symbol_table)
    ast_root = compiler.expr()
    
    print("\n--- Output Three Address Code (TAC) ---")
    compiler.generate_tac(ast_root)
except Exception as e:
    print(f"Error: {e}")

Input: a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2


1.Mengapa fungsi power() harus dipanggil di dalam term(), bukan sebaliknya? Jelaskan kaitannya dengan Operator Precedence.? Fungsi power() harus dipanggil di dalam term() karena dalam hierarki matematika, eksponen memiliki prioritas (precedence) yang lebih tinggi daripada perkalian dan pembagian. Dalam teknik Recursive Descent Parsing, fungsi yang menangani operator dengan prioritas lebih tinggi harus berada lebih "dalam" atau lebih jauh dari titik awal evaluasi ekspresi. Dengan memanggil power() di dalam term(), parser memastikan bahwa operasi pangkat diselesaikan terlebih dahulu sebelum hasilnya dikalikan atau dibagi oleh operan lain di tingkat term(). Jika urutannya dibalik, parser akan mencoba melakukan perkalian sebelum mengetahui apakah ada eksponen yang melekat pada angka tersebut, yang mana akan melanggar aturan standar matematika (PEMDAS/BODMAS).

2.Apa yang terjadi pada fase Analisis Semantik jika variabel z digunakan dalam kode sumber tetapi tidak ada di symbol_table? Pada fase Analisis Semantik, jika compiler menemukan variabel z yang digunakan dalam kode sumber namun tidak tercatat di dalam symbol_table, maka akan terjadi Semantic Error berupa Undeclared Identifier. Tabel simbol berfungsi sebagai pusat referensi untuk memverifikasi atribut identitas variabel, seperti tipe data dan ruang lingkup (scope). Ketika penganalisis semantik melakukan pengecekan keberadaan (lookup), dan hasilnya nihil, proses kompilasi biasanya akan terhenti atau mencatat pesan kesalahan karena komputer tidak mengetahui berapa banyak memori yang harus dialokasikan atau operasi apa yang valid untuk dilakukan pada z.

3.Jelaskan mengapa dalam TAC, instruksi untuk a ^ 2 harus muncul sebelum instruksi untuk +.? Dalam pembuatan Three-Address Code (TAC) untuk ekspresi seperti a ^ 2 + b, instruksi untuk a ^ 2 wajib muncul sebelum instruksi untuk + karena prinsip ketergantungan data dan urutan eksekusi linier. TAC memecah ekspresi kompleks menjadi urutan instruksi sederhana yang masing-masing hanya memiliki maksimal tiga alamat. Karena operator + membutuhkan hasil dari a ^ 2 sebagai salah satu inputnya, maka nilai perpangkatan tersebut harus dihitung terlebih dahulu dan disimpan ke dalam variabel sementara (misalnya t1 = a ^ 2). Tanpa menyelesaikan instruksi pangkat terlebih dahulu, instruksi penjumlahan tidak akan memiliki nilai operan yang valid untuk diproses.